In [61]:
! pip install pymysql
! pip install neo4j
! pip install python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)


In [1]:
import pymysql
from neo4j import GraphDatabase, exceptions
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
conn = pymysql.connect(
    host="localhost", 
    user="root", 
    password=os.getenv("MYSQL_PASSWORD"), 
    database="appdbproj", 
    cursorclass=pymysql.cursors.DictCursor, 
    port=3306
    )

In [4]:
def print_menu():
    print("""
Conference Management
--------------------------
MENU
====
1 - View Speakers & Sessions
2 - View Attendees by Company
3 - Add New Attendee
4 - View Connected Attendees
5 - Add Attendee Connection
6 - View Rooms
x - Exit application""")
    choice = input("Choice: ")
    return choice


neo4jdriver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "neo4jneo4j")  
    )


def get_names(tx, attendeeID):
    query = """
        MATCH (a:Attendee {AttendeeID: $ID})-[:CONNECTED_TO]-(b:Attendee)
        RETURN b.AttendeeID AS AttendeeID
    """
    results = tx.run(query, ID=attendeeID)
    IDs = []
    for result in results:
        IDs.append(result["AttendeeID"])
    return IDs


def sql_queries():
    
    while True:
        choice = print_menu()

        if choice == "1":
            speaker_name = input("Enter speaker name: ")
            query = """SELECT a.attendeeName AS Name, s.sessionTitle AS session, rm.roomName AS room 
                    FROM attendee AS a 
                    RIGHT JOIN registration AS r ON a.attendeeID = r.attendeeID 
                    RIGHT JOIN session AS s ON r.sessionID = s.sessionID 
                    RIGHT JOIN room AS rm ON s.roomID = rm.roomID
                    WHERE a.attendeeName LIKE %s"""
            
            with conn.cursor() as cursor:
                cursor.execute(query, (f"%{speaker_name}%",))
                subjects = cursor.fetchall()
                if not subjects:
                    print("No speaker found of that name")
                else:
                    print("""
-----------------------------------------------""")
                    for s in subjects:
                        print(f"{s['Name']}  |  {s['session']}  |  {s['room']}")
                    print("""-----------------------------------------------
                          """)
        
        if choice == "2":

            with conn.cursor() as cursor:
                
                while True:
                    
                    companyID = input("Enter company ID: ")

                    cursor.execute("SELECT companyName FROM company WHERE companyID = %s", (companyID,))
                    company = cursor.fetchone()
                    
                    
                    if not company:
                        print(f"Company with ID {companyID} does not exist")
                        continue

                    cursor.execute("SELECT * FROM attendee WHERE attendeeCompanyID = %s", (companyID,))
                    attendees = cursor.fetchall()

                    if not attendees:
                        print(f"No attendees found for {company["companyName"]}")
                        continue
                    
                    query = """SELECT a.attendeeName AS Name, a.attendeeDOB AS DOB, s.sessionTitle AS sessionTitle, 
                    s.speakerName AS sessionSpeaker, rm.roomName AS roomName
                    FROM attendee AS a 
                    RIGHT JOIN registration AS r ON a.attendeeID = r.attendeeID 
                    RIGHT JOIN session AS s ON r.sessionID = s.sessionID 
                    RIGHT JOIN room AS rm ON s.roomID = rm.roomID
                    RIGHT JOIN company AS c ON a.attendeeCompanyID = c.companyID
                    WHERE c.companyID = %s"""

                    cursor.execute(query, (companyID,))
                    subjects = cursor.fetchall()
                    print()
                    print("-----------------------------------------------")
                    print(company["companyName"])
                    for s in subjects:
                        print(f"{s['Name']}  |  {s['DOB']}  |  {s['sessionTitle']}  |  {s['sessionSpeaker']}  |  {s['roomName']}")
                    print("-----------------------------------------------")
                    print()
                    break

        if choice == "3":
            print("""
Add New Attendee
----------------""")
            attendeeID = input("Enter attendee ID: ")
            attendee_name = input("Enter attendee's first name and surname: ")
            attendeeDOB = input("Enter the date of birth of attendee (YYYY-MM-DD): ")
            attendee_gender = input("Enter the gender of the attendee: ")
            attendee_company = input("Enter the ID of the attendee's company: ")

            with conn.cursor() as cursor:

                cursor.execute("SELECT attendeeID FROM attendee WHERE attendeeID = %s", (attendeeID,))
                attendee = cursor.fetchone()

                if attendee:
                    print(f"*** ERROR *** Attendee ID: {attendeeID} already exist")
                
                gender_correct = attendee_gender != "Male" or attendee_gender != "Female"

                if gender_correct:
                    print("*** ERROR *** Gender must be Male/Female")
                
                cursor.execute("SELECT companyName FROM company WHERE companyID = %s", (attendee_company,))
                company = cursor.fetchone()
                    
                if not company:
                    print(f"*** ERROR *** Company ID: {attendee_company} does not exist")
                    
                

                if not attendee and company and gender_correct:

                    try:
                    
                        query = "INSERT INTO attendee VALUES (%s, %s, %s, %s, %s)"


                        cursor.execute(query, (attendeeID, attendee_name, attendeeDOB, attendee_gender, attendee_company))
                        conn.commit()  
                        print(f"Attendee {attendee_name} successfully added.")

                        cursor.execute("SELECT * FROM attendee WHERE attendeeID = %s", (attendeeID,))
                        new_attendee = cursor.fetchone()


                        print("-----------------------------------------------")
                        print(f"ID: {new_attendee['attendeeID']}  |  Name: {new_attendee['attendeeName']}  |  DOB: {new_attendee['attendeeDOB']}  |  Gender: {new_attendee['attendeeGender']}  |  Company ID: {new_attendee['attendeeCompanyID']}")
                        print("-----------------------------------------------")
                        print()

                    except pymysql.Error as e:
                        print(f"*** Error *** {e}")
                        conn.rollback()


        if choice == "4":

            while True:
                attendeeID = input("Enter attendee ID: ")
                if attendeeID.isdigit():
                    attendeeID = int(attendeeID)
                    break
                else:
                    print("*** ERROR *** Invalid attendee ID")



            with neo4jdriver.session(database="appdbprojneo4j") as session:
                
                connected_attendees = session.execute_read(get_names, attendeeID)



            query = """SELECT attendeeID AS ID, attendeeName AS Name 
                FROM attendee 
                WHERE attendeeID IN %s"""


            with conn.cursor() as cursor:
                cursor.execute("""SELECT attendeeName AS Name 
                FROM attendee 
                WHERE attendeeID = %s""", attendeeID,)
                attendee = cursor.fetchone()

                if not connected_attendees and attendee:
                    print("No connections")

                elif not connected_attendees and not attendee:
                    print("*** ERROR *** Attendee does not exist")


                else:
                    cursor.execute(query, (tuple(connected_attendees),))
                    connected_attendees = cursor.fetchall()
                    print(f"""
-----------------------------------------------
Attendee Name: {attendee["Name"]}
-----------------------------------------------
These attendees are connected:""")
                    for attendee in connected_attendees:
                        print(f"{attendee['ID']}  |  {attendee['Name']}")
                    print("-----------------------------------------------")
                    print()

        if choice == "x":
            print("Exiting...")
            break

In [5]:
sql_queries()


Conference Management
--------------------------
MENU
====
1 - View Speakers & Sessions
2 - View Attendees by Company
3 - Add New Attendee
4 - View Connected Attendees
5 - Add Attendee Connection
6 - View Rooms
x - Exit application
Exiting...
